# DATA2001 Full Workflow


## 0. Team Information


| name | unikey | SA4 Zone |
|------|--------|----------|
| xuejian fang | xfan0282 | Sydney - City and Inner South |
| xuanhao yu | xuyu8020 | Sydney - Northern Beaches |
|
|

## 1. Environment and Configuration

This section imports the dependencies and project interfaces used by the rest of the workflow, loads the YAML configuration, and creates the database engine for later steps.

> Prerequisites: follow the README first to prepare the environment. Make sure the local PostgreSQL/PostGIS container is running, run `uv sync` from the project root to create the virtual environment, and select that virtual environment as the Python kernel for this notebook.


In [10]:
from pathlib import Path

import pandas as pd
import plotly.express as px

from data2001.config import load_settings
from data2001.db.engine import create_engine_from_settings
from data2001.pipeline import execute_workflow_steps
from data2001.task1_cleaning.workflow import run_task1_cleaning
from data2001.task1_statistics.workflow import run_all_task1_statistics
from data2001.task4.queries import (
    expected_report_figure_paths,
    load_api_extraction_summary,
    load_correlation_results,
    load_correlation_summary,
    load_index_summary,
    load_poi_group_counts,
    load_poi_points,
    load_sa2_scores,
    load_schema_summary,
    load_score_income,
    load_score_input_summary,
    load_spatial_join_summary,
    load_table_counts,
)
from data2001.task4.charts import (
    build_bottom_sa2_bar,
    build_poi_group_distribution,
    build_score_histogram,
    build_score_income_scatter,
    build_top_sa2_bar,
)
from data2001.task4.maps import build_poi_point_scatter_map, build_score_choropleth_map
from data2001.task4.tables import build_sa4_summary_table, build_top_bottom_table


settings = load_settings("configs/local.yaml")
engine = create_engine_from_settings(settings.database)

settings


Settings(database=DatabaseSettings(driver='postgresql+psycopg', host='localhost', port=5432, database='data2001', user='data2001', password='data2001', schema_name='data2001'), api=APISettings(layers={'poi': LayerSettings(url='https://maps.six.nsw.gov.au/arcgis/rest/services/public/NSW_POI/MapServer/0/query', geometry_type='esriGeometryPoint', expected_srid=4283, expected_fields=['objectid', 'topoid', 'poigroup', 'poitype', 'poiname', 'poilabel', 'poilabeltype', 'poialtlabel', 'poisourcefeatureoid', 'accesscontrol', 'startdate', 'enddate', 'lastupdate', 'msoid', 'centroidid', 'shapeuuid', 'changetype', 'processstate', 'urbanity'], out_fields=['objectid', 'topoid', 'poigroup', 'poitype', 'poiname', 'poilabel', 'poilabeltype', 'poialtlabel', 'poisourcefeatureoid', 'accesscontrol', 'startdate', 'enddate', 'lastupdate', 'msoid', 'centroidid', 'shapeuuid', 'changetype', 'processstate', 'urbanity']), 'sa2': LayerSettings(url='https://geo.abs.gov.au/arcgis/rest/services/ASGS2021/SA2/FeatureSe

## 1.1 Selected SA4 Configuration

This section reads each team member's SA4 configuration from the config file. Later steps only process the selected SA4 areas.


In [11]:
selected_sa4_df = pd.DataFrame(
    sorted(settings.task2_import.selected_sa4_by_member.items()),
    columns=["member", "selected_sa4"],
)

config_summary = pd.DataFrame(
    [
        {"setting": "task2_import.crawl_scope", "value": settings.task2_import.crawl_scope},
        {"setting": "task3_score.score_universe", "value": settings.task3_score.score_universe},
        {"setting": "dashboard.url", "value": "https://kscii.tech"},
        {"setting": "repository.url", "value": "https://github.sydney.edu.au/xfan0282/data2001-group-assignment"},
    ]
)

display(config_summary)
display(selected_sa4_df)


,setting,value
0,task2_import.crawl_scope,selected_sa4
1,task3_score.score_universe,selected_sa4
2,dashboard.url,https://kscii.tech
3,repository.url,https://github.sydney.edu.au/xfan0282/data2001-group-assignment


,member,selected_sa4
0,dabi0142,Sydney - Parramatta
1,jzho0172,Sydney - North Sydney and Hornsby
2,xfan0282,Sydney - City and Inner South
3,xuyu8020,Sydney - Eastern Suburbs


## 2. Task 1: CSV Loading, Cleaning, and Derived Statistics

This section applies the shared cleaning workflow to the original CSV data.

> [TODO] Each team member should briefly explain the cleaning step they implemented.


In [12]:
from data2001.common.paths import resolve_project_path

raw_task1_csv = resolve_project_path(settings.outputs.raw_task1_csv)
cleaned_task1_csv = resolve_project_path(settings.outputs.processed_task1_cleaned_csv)

raw_task1_df = pd.read_csv(raw_task1_csv)
cleaned_task1_df = run_task1_cleaning(raw_task1_csv, cleaned_task1_csv)
statistics_df = run_all_task1_statistics(cleaned_task1_df)

display(raw_task1_df.head())
display(cleaned_task1_df.head())
display(statistics_df)

Statistics workflow errors:
- dabi0142_1: IndexError: single positional indexer is out-of-bounds
- dabi0142_2: IndexError: single positional indexer is out-of-bounds
- dabi0142_3: TypeError: StatisticResult.__init__() got an unexpected keyword argument 'statistic'
- dabi0142_4: TypeError: StatisticResult.__init__() got an unexpected keyword argument 'statistic'
- dabi0142_5: TypeError: StatisticResult.__init__() got an unexpected keyword argument 'statistic'


,Measure Code,Parent Description,Description,2011,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,ERP_P_20,Estimated resident population - year ended 30 June,Estimated resident population (no.),NaN,NaN,NaN,NaN,NaN,8046748.0,8110610.0,8097062.0,8166704.0,8341199.0,8479314.0,NaN
1,ERP_21,Estimated resident population - year ended 30 June,Population density (persons/km2),NaN,NaN,NaN,NaN,NaN,10.0,10.1,10.1,10.2,10.4,10.6,NaN
2,ERP_M_20,Estimated resident population - year ended 30 June,Estimated resident population - males (no.),NaN,NaN,NaN,NaN,NaN,3999452.0,4030710.0,4025393.0,4059763.0,4149032.0,4217861.0,NaN
3,ERP_F_20,Estimated resident population - year ended 30 June,Estimated resident population - females (no.),NaN,NaN,NaN,NaN,NaN,4047296.0,4079900.0,4071669.0,4106941.0,4192167.0,4261453.0,NaN
4,ERP_19,Estimated resident population - year ended 30 June,Median age - males (years),NaN,NaN,NaN,NaN,NaN,36.8,37.2,37.7,37.7,37.5,37.5,NaN


,measure_code,parent_description,description,unit,year,value
0,CENSUS_34,Aboriginal and Torres Strait Islander Peoples - Census,Aboriginal and Torres Strait Islander Peoples,no.,2011,172620.0
1,CENSUS_2,Aboriginal and Torres Strait Islander Peoples - Census,Aboriginal and Torres Strait Islander Peoples,%,2011,2.5
2,CENSUS_15,Religious affiliation - Census,Buddhism,%,2011,2.9
3,CENSUS_16,Religious affiliation - Census,Christianity,%,2011,64.5
4,CENSUS_17,Religious affiliation - Census,Hinduism,%,2011,1.7


,member,statistic_id,title,value,unit,description
0,xuyu8020,xuyu8020-1,"Age dependency ratio, 2024",54.44,dependents per 100 working-age persons,"In 2024, NSW had about 54.44 people aged outside 15-64 for every 100 people of working age. This statistic compares the estimated resident population with the working-age population."
1,xuyu8020,xuyu8020-2,"Sex ratio, 2024",98.98,males per 100 females,"In 2024, NSW had approximately 98.98 males per 100 females. This indicates that the female population was slightly larger than the male population."
2,xuyu8020,xuyu8020-3,"Median total income growth, 2018-2022",15.16,%,"Median total income excluding government pensions and allowances increased from $47,852 in 2018 to $55,105 in 2022. This represents a growth rate of 15.16%."
3,xuyu8020,xuyu8020-4,"Mean-to-median total income gap, 2022",37.03,% above median,"In 2022, mean total income ($75,511) was 37.03% higher than median total income ($55,105). This provides a simple indication that the income distribution was right-skewed."
4,xuyu8020,xuyu8020-5,"Business net entry rate, 2024",2.95,% of total businesses,"In 2024, NSW recorded 149,226 business entries and 122,778 business exits. The net increase was equivalent to 2.95% of the total number of businesses."
5,xfan0282,xfan0282-1,"Apartment share increase among occupied private dwellings, 2011-2021",2.90,percentage points,"Apartment share increased 2.90pp to 21.72% (2011-2021), indicating residential structure shift."
6,xfan0282,xfan0282-2,"Work-from-home share growth, 2016-2021",6.42,times,Work-from-home share grew 6.42x from 4.82% to 30.98% (2016-2021).
7,xfan0282,xfan0282-3,"Public transport commute share drop, 2016-2021",11.98,percentage points,Public transport commute share dropped 11.98pp from 15.98% to 4.00% (2016-2021).
8,xfan0282,xfan0282-4,"Occupation commute distance gap, 2016",5.30,km,Commute distance gap in 2016: 5.3 km (range 14.7-20.0 km across occupations).
9,xfan0282,xfan0282-5,"Rent stress relative to mortgage stress, 2021",2.05,times,Rent stress (35.5%) is 2.05x mortgage stress (17.3%) in 2021.


## 2.1 Task 1 Key Findings

This section displays the statistic values and the explanations attached to those statistics.

> [TODO] Each team member should add their explanation text in `statistics.py`.


In [13]:
xfan0282_statistics = (
    statistics_df[statistics_df["member"] == "xfan0282"]
    .sort_values("statistic_id")
    .reset_index(drop=True)
)

pd.set_option('display.max_colwidth', 100)  # Set the maximum display width for pandas text columns to 100 characters.
display(
    xfan0282_statistics[
        ["statistic_id", "title", "value", "unit", "description"]
    ]
)


,statistic_id,title,value,unit,description
0,xfan0282-1,"Apartment share increase among occupied private dwellings, 2011-2021",2.90,percentage points,"Apartment share increased 2.90pp to 21.72% (2011-2021), indicating residential structure shift."
1,xfan0282-2,"Work-from-home share growth, 2016-2021",6.42,times,Work-from-home share grew 6.42x from 4.82% to 30.98% (2016-2021).
2,xfan0282-3,"Public transport commute share drop, 2016-2021",11.98,percentage points,Public transport commute share dropped 11.98pp from 15.98% to 4.00% (2016-2021).
3,xfan0282-4,"Occupation commute distance gap, 2016",5.30,km,Commute distance gap in 2016: 5.3 km (range 14.7-20.0 km across occupations).
4,xfan0282-5,"Rent stress relative to mortgage stress, 2021",2.05,times,Rent stress (35.5%) is 2.05x mortgage stress (17.3%) in 2021.


In [28]:
xuyu8020_statistics = (
    statistics_df[statistics_df["member"] == "xuyu8020"]
    .sort_values("statistic_id")
    .reset_index(drop=True)
)

pd.set_option("display.max_colwidth", 200)

display(
    xuyu8020_statistics[
        ["statistic_id", "title", "value", "unit", "description"]
    ]
    .style
    .set_properties(**{
        "text-align": "left"
    })
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "left")
            ]
        }
    ])
)

,statistic_id,title,value,unit,description
0,xuyu8020-1,"Age dependency ratio, 2024",54.440000,dependents per 100 working-age persons,"In 2024, NSW had about 54.44 people aged outside 15-64 for every 100 people of working age. This statistic compares the estimated resident population with the working-age population."
1,xuyu8020-2,"Sex ratio, 2024",98.980000,males per 100 females,"In 2024, NSW had approximately 98.98 males per 100 females. This indicates that the female population was slightly larger than the male population."
2,xuyu8020-3,"Median total income growth, 2018-2022",15.160000,%,"Median total income excluding government pensions and allowances increased from $47,852 in 2018 to $55,105 in 2022. This represents a growth rate of 15.16%."
3,xuyu8020-4,"Mean-to-median total income gap, 2022",37.030000,% above median,"In 2022, mean total income ($75,511) was 37.03% higher than median total income ($55,105). This provides a simple indication that the income distribution was right-skewed."
4,xuyu8020-5,"Business net entry rate, 2024",2.950000,% of total businesses,"In 2024, NSW recorded 149,226 business entries and 122,778 business exits. The net increase was equivalent to 2.95% of the total number of businesses."


## 3. Data Source Summary

This section lists the data sources used by the downstream database import steps and their API endpoints.


In [15]:
data_sources = pd.DataFrame(
    [
        {"source": "NSW CSV", "endpoint": "data/raw/task1/raw_data.csv"},
        {"source": "NSW POI", "endpoint": "https://maps.six.nsw.gov.au/arcgis/rest/services/public/NSW_POI/MapServer/0/query"},
        {"source": "SA2 Boundaries", "endpoint": "https://geo.abs.gov.au/arcgis/rest/services/ASGS2021/SA2/FeatureServer/0/query"},
        {"source": "SA4 Boundaries", "endpoint": "https://geo.abs.gov.au/arcgis/rest/services/ASGS2021/SA4/MapServer/0/query"},
        {"source": "SA2 Population", "endpoint": "https://geo.abs.gov.au/arcgis/rest/services/Hosted/ABS_Population_and_people_by_2021_SA2_Nov_2023/FeatureServer/1/query"},
        {"source": "SA2 Income", "endpoint": "https://geo.abs.gov.au/arcgis/rest/services/Hosted/Personal_Income_in_Australia_2022_23_SA2_2021/FeatureServer/0/query"},
    ]
)
pd.set_option('display.max_colwidth', 150)
display(data_sources)

,source,endpoint
0,NSW CSV,data/raw/task1/raw_data.csv
1,NSW POI,https://maps.six.nsw.gov.au/arcgis/rest/services/public/NSW_POI/MapServer/0/query
2,SA2 Boundaries,https://geo.abs.gov.au/arcgis/rest/services/ASGS2021/SA2/FeatureServer/0/query
3,SA4 Boundaries,https://geo.abs.gov.au/arcgis/rest/services/ASGS2021/SA4/MapServer/0/query
4,SA2 Population,https://geo.abs.gov.au/arcgis/rest/services/Hosted/ABS_Population_and_people_by_2021_SA2_Nov_2023/FeatureServer/1/query
5,SA2 Income,https://geo.abs.gov.au/arcgis/rest/services/Hosted/Personal_Income_in_Australia_2022_23_SA2_2021/FeatureServer/0/query


## 4. Database Schema and Indexes

This section initializes and checks the database, then displays the current database schema and indexing design.


In [16]:
db_setup_summary = execute_workflow_steps(engine, settings, ["init_db", "check_db"], title="Database setup")
db_setup_summary

{'init_db': 'done',
 'postgis_version': '3.4 USE_GEOS=1 USE_PROJ=1 USE_STATS=1',
 'table_count': 7,
 'missing_tables': '[]',
 'bad_srids': '[]'}

In [17]:
schema_summary = load_schema_summary(engine, settings)
index_summary = load_index_summary(engine, settings)

display(load_table_counts(engine, settings))
display(schema_summary.head(20))
display(index_summary)

,table_name,row_count
0,sa4,4
1,sa2,108
2,poi_clean,8926
3,sa2_poi,7265
4,sa2_income,108
5,sa2_score,103
6,score_income_correlation,2


,table_name,column_name,data_type,is_nullable,column_default,constraint_type
0,poi_clean,objectid,bigint,NO,NaN,PRIMARY KEY
1,poi_clean,topoid,bigint,YES,NaN,NaN
2,poi_clean,poigroup_code,smallint,YES,NaN,NaN
3,poi_clean,poigroup_name,text,YES,NaN,NaN
4,poi_clean,poitype,text,YES,NaN,NaN
5,poi_clean,poiname,text,YES,NaN,NaN
6,poi_clean,poilabel,text,YES,NaN,NaN
7,poi_clean,poilabeltype,text,YES,NaN,NaN
8,poi_clean,poialtlabel,text,YES,NaN,NaN
9,poi_clean,poisourcefeatureoid,bigint,YES,NaN,NaN


,table_name,index_name,definition
0,poi_clean,idx_poi_clean_geometry_gist,CREATE INDEX idx_poi_clean_geometry_gist ON data2001.poi_clean USING gist (geometry)
1,poi_clean,idx_poi_clean_poigroup,CREATE INDEX idx_poi_clean_poigroup ON data2001.poi_clean USING btree (poigroup_code)
2,poi_clean,idx_poi_clean_topoid,CREATE INDEX idx_poi_clean_topoid ON data2001.poi_clean USING btree (topoid)
3,poi_clean,poi_clean_pkey,CREATE UNIQUE INDEX poi_clean_pkey ON data2001.poi_clean USING btree (objectid)
4,sa2,idx_sa2_geometry_gist,CREATE INDEX idx_sa2_geometry_gist ON data2001.sa2 USING gist (geometry)
5,sa2,idx_sa2_population,CREATE INDEX idx_sa2_population ON data2001.sa2 USING btree (population)
6,sa2,idx_sa2_sa4_code,CREATE INDEX idx_sa2_sa4_code ON data2001.sa2 USING btree (sa4_code)
7,sa2,sa2_pkey,CREATE UNIQUE INDEX sa2_pkey ON data2001.sa2 USING btree (sa2_code)
8,sa2_income,idx_sa2_income_earners,CREATE INDEX idx_sa2_income_earners ON data2001.sa2_income USING btree (income_earners_2022_23)
9,sa2_income,idx_sa2_income_median_income,CREATE INDEX idx_sa2_income_median_income ON data2001.sa2_income USING btree (median_income_2022_23)


## 5. Task 2: API Extraction and Crawl Plan

This section calls `plan-import`. The planning step uses API metadata to estimate how many SA4, SA2, and bbox requests will be processed under the current configuration.

It then runs the actual API extraction and writes the raw responses to disk. This usually takes about 60 seconds, and the API responses are cached locally as JSON files.

After that, the workflow cleans the API data, deduplicates records from bbox-based requests, and loads the results into the local database.

Finally, it shows the local JSON file summary and database state.


In [18]:

plan_result = execute_workflow_steps(
    engine,
    settings,
    ["plan_import"],
    title="Plan import",
)

pd.DataFrame(
    [
        {
            "scope": plan_result["crawl_scope"],
            "sa4_count": plan_result["sa4_count"],
            "sa2_count": plan_result["sa2_count"],
            "sa2_bbox_requests": plan_result["sa2_bbox_requests"],
        }
    ]
)


,scope,sa4_count,sa2_count,sa2_bbox_requests
0,selected_sa4,4,108,108


In [19]:
# This cell runs API extraction and database loading, so it may take a while.
task2_summary = execute_workflow_steps(
    engine,
    settings,
    ["import_boundaries", "import_poi", "import_income"],
    title="Task 2 data import",
)
task2_summary

{'sa4': 4,
 'sa2': 108,
 'population': 2473,
 'sa2_bbox_requests': 108,
 'raw_responses': 108,
 'raw_features_seen': 15209,
 'clean_features_seen': 8926,
 'fetch_seconds': 19.982440699975996,
 'persist_seconds': 0.3708790999808116,
 'clean_seconds': 1.3804348000048776,
 'load_seconds': 0.1418030000058934,
 'income': 2454}

In [20]:
display(load_api_extraction_summary(settings))
display(load_table_counts(engine, settings))

,response_dir,response_file_count,features_jsonl,raw_feature_rows,features_file_exists,features_file_size_mb
0,D:\data2001\data2001-group-assignment\data\raw\poi_api\responses,108,D:\data2001\data2001-group-assignment\data\raw\poi_api\features.jsonl,15209,True,8.04


,table_name,row_count
0,sa4,4
1,sa2,108
2,poi_clean,8926
3,sa2_poi,7265
4,sa2_income,108
5,sa2_score,103
6,score_income_correlation,2


## 6. Spatial Join Evidence

After POI records are fetched from the API using SA2 bboxes, this workflow uses PostGIS `ST_Covers(sa2.geometry, poi_clean.geometry)` to determine which SA2 polygon each POI belongs to and to remove duplicate assignments.

If a POI lies on the boundary of two SA2 polygons, the workflow assigns it to the SA2 with the lowest `sa2_code` in ascending order.

This section displays the POI assignment summary statistics.


In [21]:
spatial_join_summary = load_spatial_join_summary(engine, settings)
display(spatial_join_summary)

,clean_poi,assigned_poi,unassigned_poi,boundary_duplicate_candidates,assignment_rows
0,8926,7265,1661,0,7265


## 7. Task 3: Score Calculation

The score is calculated from the POI count using a z-score and sigmoid transformation: `score_100 = sigmoid(z_poi) * 100`.

This section first displays the score input summary, then runs the score and correlation step.


In [22]:
display(load_score_input_summary(engine, settings))

score_summary = execute_workflow_steps(engine, settings, ["compute_score"], title="Task 3 score")
score_summary

,sa2_count,total_poi,mean_poi_count,std_poi_count,min_poi_count,max_poi_count,below_min_population,missing_population
0,108,7265,67.268519,50.826248,3,354,5,1


{'scores': 103, 'correlations': 2}

In [23]:
scores = load_sa2_scores(
    engine,
    settings,
)
display(scores.head())
display(build_top_bottom_table(scores, n=settings.charts.top_n))

,sa2_code,sa2_name,sa4_code,sa4_name,population,poi_count,mean_poi_count,std_poi_count,z_poi,score_raw,score_100,geometry
0,117011320,Banksmeadow,117,Sydney - City and Inner South,594.0,4,70.07767,50.375423,-1.311705,0.212202,21.220176,"{'type': 'MultiPolygon', 'coordinates': [[[[151.208061638, -33.954059221], [151.208160078, -33.953929501], [151.208503219, -33.953330991], [151.20..."
1,117011321,Botany,117,Sydney - City and Inner South,13254.0,36,70.07767,50.375423,-0.676474,0.337049,33.704870,"{'type': 'MultiPolygon', 'coordinates': [[[[151.189644799, -33.948142551], [151.189187709, -33.947136501], [151.189000208, -33.946685821], [151.18..."
2,117031638,Camperdown - Darlington,117,Sydney - City and Inner South,8452.0,61,70.07767,50.375423,-0.180200,0.455071,45.507142,"{'type': 'MultiPolygon', 'coordinates': [[[[151.172669888, -33.890466691], [151.172847939, -33.889499961], [151.173060728, -33.888359991], [151.17..."
3,117031639,Chippendale,117,Sydney - City and Inner South,8237.0,11,70.07767,50.375423,-1.172748,0.236359,23.635866,"{'type': 'MultiPolygon', 'coordinates': [[[[151.193987418, -33.886566411], [151.194404908, -33.885604821], [151.194450098, -33.885503551], [151.19..."
4,117031329,Darlinghurst,117,Sydney - City and Inner South,10617.0,49,70.07767,50.375423,-0.418412,0.396897,39.689686,"{'type': 'MultiPolygon', 'coordinates': [[[[151.212268848, -33.876327031], [151.212319178, -33.875927281], [151.212397698, -33.875310331], [151.21..."


,rank_group,sa2_code,sa2_name,sa4_name,poi_count,score_100,population
0,top,117021328,Sydenham - Tempe - St Peters,Sydney - City and Inner South,354,99.644602,8395.0
1,top,117031644,Sydney (North) - Millers Point,Sydney - City and Inner South,304,99.046854,8181.0
2,top,121031408,Lindfield - Roseville,Sydney - North Sydney and Hornsby,172,88.321717,24661.0
3,top,121021404,Berowra - Brooklyn - Cowan,Sydney - North Sydney and Hornsby,163,86.349142,11758.0
4,top,118011346,Rose Bay - Vaucluse - Watsons Bay,Sydney - Eastern Suburbs,135,78.393891,11847.0
5,top,125041489,North Parramatta,Sydney - Parramatta,132,77.368151,22919.0
6,top,121031407,Gordon - Killara,Sydney - North Sydney and Hornsby,127,75.583194,22435.0
7,top,121031412,Wahroonga (East) - Warrawee,Sydney - North Sydney and Hornsby,127,75.583194,18150.0
8,top,121011683,Castle Cove - Northbridge,Sydney - North Sydney and Hornsby,126,75.214987,13336.0
9,top,121031411,Turramurra,Sydney - North Sydney and Hornsby,124,74.467478,20170.0


## 8. Results Analysis

The following visualisations explain the score distribution, spatial trends, Top/Bottom SA2 areas, and POI characteristics:

| Chart | Description |
|------|------|
| Score histogram | Overall distribution shape of SA2 scores, including normality, skewness, and kurtosis. |
| Top N SA2 bar chart | Ranking and exact score values for the highest-scoring areas. |
| Bottom N SA2 bar chart | Ranking and exact score values for the lowest-scoring areas. |
| POI group distribution | Count composition across POI categories such as business, education, and health. |
| Score map | Spatial distribution of SA2 scores using colour encoding. |
| SA4 summary table | Aggregated statistical summary by SA4. |
| POI point map | Spatial distribution of raw POI point locations. |

> [TODO] Add analysis and interpretation for the information shown in each chart.


In [24]:
poi_groups = load_poi_group_counts(engine, settings)
poi_points = load_poi_points(engine, settings, limit=settings.dashboard.poi_limit)

build_score_histogram(scores, nbins=settings.charts.score_histogram_nbins).show()
build_top_sa2_bar(scores, n=settings.charts.top_n).show()
build_bottom_sa2_bar(scores, n=settings.charts.top_n).show()
build_poi_group_distribution(poi_groups).show()
build_score_choropleth_map(scores).show()

In [25]:
sa4_summary = build_sa4_summary_table(scores, poi_points)
display(sa4_summary)

# Optional: the POI point map can be large; if the browser becomes slow, display only limited data.
build_poi_point_scatter_map(poi_points).show()

,sa4_name,sa2_count,total_poi,mean_score_100,median_score_100,highest_score_sa2,highest_score_100,lowest_score_sa2,lowest_score_100
2,Sydney - North Sydney and Hornsby,26,2352,58.73,57.11,Lindfield - Roseville,88.32,Hornsby - East,33.70
3,Sydney - Parramatta,31,2086,48.38,48.97,North Parramatta,77.37,South Wentworthville,25.47
0,Sydney - City and Inner South,25,1717,43.21,39.69,Sydenham - Tempe - St Peters,99.64,Banksmeadow,21.22
1,Sydney - Eastern Suburbs,21,1110,41.64,36.88,Rose Bay - Vaucluse - Watsons Bay,78.39,Kingsford,25.85


## 9. Correlation Analysis

This section uses Pearson correlation as the main test and Spearman correlation as a rank-based robustness check.


In [26]:
score_income = load_score_income(engine, settings)
correlation_results = load_correlation_results(engine, settings)

display(load_correlation_summary(engine, settings))
display(score_income.head())
build_score_income_scatter(score_income).show()

,method,statistic,p_value,n,alpha,is_significant,created_at,interpretation
0,pearson,0.135687,0.176077,101,0.05,False,2026-05-16 17:57:52.360734+00:00,not statistically significant
1,spearman,0.156345,0.118450,101,0.05,False,2026-05-16 17:57:52.360734+00:00,not statistically significant


,sa2_code,sa2_name,sa4_name,score_100,poi_count,population,median_income_2022_23,income_earners_2022_23
0,117011320,Banksmeadow,Sydney - City and Inner South,21.220176,4,594.0,81791.0,396.0
1,117011321,Botany,Sydney - City and Inner South,33.704870,36,13254.0,74415.0,9296.0
2,117031638,Camperdown - Darlington,Sydney - City and Inner South,45.507142,61,8452.0,63833.0,6461.0
3,117031639,Chippendale,Sydney - City and Inner South,23.635866,11,8237.0,45733.0,6577.0
4,117031329,Darlinghurst,Sydney - City and Inner South,39.689686,49,10617.0,74948.0,9834.0


## 10. Report Figures

This section exports the PNG figures used by the report and lists the expected output paths. If Chrome/Kaleido support is missing, run `uv run plotly_get_chrome` first.


In [27]:
figure_summary = execute_workflow_steps(engine, settings, ["export_charts"], title="Task 4 report figures")
display(figure_summary)
display(expected_report_figure_paths(settings))

KeyboardInterrupt: 

## 11. Dashboard and Repository Links

Deployed dashboard: https://kscii.tech

Repository: https://github.sydney.edu.au/xfan0282/data2001-group-assignment

Optional local dashboard command:

```bash
uv run data2001 dashboard
```


## 12. Limitations

> [TODO] List the project limitations.


## 13. Full Reproducibility Command

To rerun the full workflow from an empty database, use the following commands:

```bash
uv sync
podman compose up -d
uv run data2001 init-db
uv run data2001 run-workflow
uv run data2001 generate-figures
# optional local dashboard
uv run data2001 dashboard
```
